In [1]:
import torch 
import matplotlib.pyplot as plt
import random
import numpy as np
from matplotlib.colors import ListedColormap
import dill
from sparse_generalization.envs.box_world.env import BoxWorldEnv
from sparse_generalization.envs.box_world.wrappers import make_env
from minigrid.wrappers import FullyObsWrapper
import gymnasium as gym
import cv2
import matplotlib.pyplot as plt
from torchmetrics.classification.accuracy import BinaryAccuracy
gym.register('BoxWorldEnv-v1', BoxWorldEnv)

%load_ext autoreload
%autoreload 2

c:\Users\garga\Documents\Uni\Thesis\sparse-generalization\.venv\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
c:\Users\garga\Documents\Uni\Thesis\sparse-generalization\.venv\Lib\site-packages\gymnasium\envs\registration.py:636: UserWarning: WARN: Overriding environment BoxWorldEnv-v1 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")


In [11]:
with open('../results/mlp_run_simple_24_Feb_2026__01h52m.pl', 'rb') as file:
    results = dill.load(file)

In [12]:
results[2]

{'train_loss': [0.6989783346652985,
  0.6920813649892807,
  0.6935216754674911,
  0.6924620181322098,
  0.6941561222076416,
  0.6921784311532975,
  0.6907735586166381,
  0.68946872651577,
  0.6873125195503235,
  0.6875702291727066,
  0.6839450657367706,
  0.6874796897172928,
  0.6854657381772995,
  0.6795766949653625,
  0.6778179734945298,
  0.675946193933487,
  0.6736634820699692,
  0.6719166308641433,
  0.6702804058790207,
  0.6679453939199448],
 'train_acc': [0.4937844663858414,
  0.5189797788858413,
  0.5180491715669632,
  0.5147748172283173,
  0.5142922788858414,
  0.5177045047283173,
  0.5215533092617989,
  0.5334903478622437,
  0.5490119487047196,
  0.5519646137952805,
  0.5650620400905609,
  0.5531364887952804,
  0.5510914534330368,
  0.5699103862047196,
  0.5766773909330368,
  0.5741268396377563,
  0.5751838237047195,
  0.5791934728622437,
  0.5865923702716828,
  0.5872817099094391],
 'train_sparse': [0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
 

In [25]:
with open('../data/box_world/balanced_dist_500.pl', 'rb') as file:
    dataset = dill.load(file)
    file.close()

In [26]:
x_train = dataset['X_train']
y_train = dataset['Y_train']
x_test_ind = dataset['X_test_ind']
y_test_ind = dataset['Y_test_ind']
x_test_ood = dataset['X_test_ood']
y_test_ood = dataset['Y_test_ood']
edges_train = dataset['edges_train']
edges_test = dataset['edges_test_ood']

In [195]:
test = edges_train[0:50]
test = test.view(50 * 1 * 4, 2, 2)
test = test[:, :, 1] * 10 + test[:, :, 0] 
test = test.view(50, 4, 2)
A = torch.zeros(50, 100, 100)
item = test[0]
A[0, item[:, 0], item[:, 1]] = 1

In [190]:
test[0]

tensor([[[16, 54],
         [11, 12],
         [51, 52],
         [27, 26]]])

In [203]:
edges_train[3]

tensor([[[[6, 7],
          [5, 8]],

         [[4, 6],
          [5, 6]],

         [[1, 7],
          [2, 7]],

         [[8, 5],
          [7, 5]]]])

In [145]:
def get_edges(obs, residual=False):
    edges = []
    # get goal-lock 
    x_goal, y_goal = torch.where(obs[:, :, 0] == 8)
    goal = torch.stack([x_goal, y_goal], dim=1).squeeze()
    edges.append(((goal[0]+1, goal[1]), (goal[0], goal[1])))
    # for each key check key + 1 in lock then a combo
    xs, ys = torch.where(x_train[0][:, :, 0] == 13)
    locks = torch.stack([xs, ys], dim=1)
    locks = locks[~(locks == torch.tensor([goal[0]+1, goal[1]])).all(dim=1)]
    
    xs, ys = torch.where(x_train[0][:, :, 0] == 12)
    keys = torch.stack([xs, ys], dim=1)
    for lock in locks:
        key = (lock[0]-1, lock[1])
        edges.append(((lock[0], lock[1]), (key)))
        keys = keys[~(keys == torch.tensor([lock[0]-1, lock[1]])).all(dim=1)]
        
    # remaining key is residual or looking from agent
    first_key = keys.squeeze()
    if not residual: 
        xs, ys = torch.where(x_train[0][:, :, 0] == 10)
        agent = torch.stack([xs, ys], dim=1).squeeze()
        edges.append(((agent[0], agent[1]), (first_key[0], first_key[1])))
    
    return edges

In [43]:
from sparse_generalization.layers.oracle import MultiHeadAttentionOracle

mha = MultiHeadAttentionOracle(embed_size=3, num_heads=3)
x_train = x_train.view(500, 100, 3).float()
mha(x_train, x_train, x_train, edges_train)[1].shape

torch.Size([500, 100, 100])